# Reentrenamiento con el dataset final de Roboflow (best.pt v2)

Proyecto Roboflow: `adrianas-workspace-n5bac / deteccion-de-sillar-3fyav` (Instance Segmentation).

Este notebook **no modifica nada en Roboflow**. Descarga la versión indicada y, en Colab:

1. Unifica las 10 clases del proyecto en las 3 que usa la app (`crack`, `humidity`, `spalling`):
   `grieta`/`fisura` → `crack`, `humedad`/`humeldad` → `humidity`, `desprendimiento` → `spalling`.
   Las imágenes con `Patologias` o clases basura (`3`, `67`, `78`, `h4`) se **excluyen** (no se
   convierten en fondo, porque contienen daño real sin clase definida).
2. Cuenta instancias por clase e imágenes de fondo (null) por split: evidencia para la tesis.
3. Entrena `yolov8s-seg.pt` (segmentación → la app mide el área en cm²/m² sobre la máscara real).
4. Evalúa en el split de **test** y verifica que `best.pt` tenga exactamente las 3 clases.

## Antes de ejecutar (en Roboflow)
- Descargar una copia de la v1 como respaldo.
- Subir 150-250 tiles de cornisas, sombras, juntas, cables, cielo y vegetación y marcarlos
  **Mark Null** en Annotate (sin eso no llegan al entrenamiento).
- **Generate → New Version** con: Auto-Orient + Resize **Fit within 640×640** (no *Stretch*),
  split **70/20/10**. Las augmentations se hacen en el entrenamiento (celda 6), así que en
  Roboflow pueden dejarse vacías.
- Guardar la API key privada de Roboflow en Colab: icono 🔑 *Secrets* → nombre
  `ROBOFLOW_API_KEY` (nunca escribirla en el notebook).

In [ ]:
# 1) Montar Google Drive y verificar GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Instalar la misma version de ultralytics usada para best.pt + cliente de Roboflow
!pip install -q ultralytics==8.4.56 roboflow

In [ ]:
# 3) Descargar la version del dataset desde Roboflow
from google.colab import userdata
from roboflow import Roboflow

VERSION = 2   # <-- numero de la version generada en Roboflow con las 2030 imagenes

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
proyecto = rf.workspace("adrianas-workspace-n5bac").project("deteccion-de-sillar-3fyav")
ds = proyecto.version(VERSION).download("yolov8", location="/content/datasets/sillar_rf", overwrite=True)
print("Descargado en:", ds.location)

In [ ]:
# 4) Unificar las clases en crack / humidity / spalling (sin tocar Roboflow)
import os, shutil, yaml
from collections import Counter

ORIGEN = ds.location
DESTINO = "/content/datasets/sillar_3clases"
CLASES = ["crack", "humidity", "spalling"]
ALIAS = {
    "crack": "crack", "grieta": "crack", "fisura": "crack",
    "humidity": "humidity", "humedad": "humidity", "humeldad": "humidity",
    "spalling": "spalling", "desprendimiento": "spalling",
}

with open(os.path.join(ORIGEN, "data.yaml")) as f:
    nombres = yaml.safe_load(f)["names"]
if isinstance(nombres, dict):
    nombres = [nombres[k] for k in sorted(nombres)]
mapa = {}
for i, n in enumerate(nombres):
    clave = str(n).strip().lower()
    mapa[i] = CLASES.index(ALIAS[clave]) if clave in ALIAS else None
    print(f"  {i:2d} {n!r:20s} -> {CLASES[mapa[i]] if mapa[i] is not None else 'EXCLUIR imagen'}")

shutil.rmtree(DESTINO, ignore_errors=True)
resumen, excluidas, es_segmentacion = {}, [], False
for split in ("train", "valid", "test"):
    dir_img = os.path.join(ORIGEN, split, "images")
    if not os.path.isdir(dir_img):
        continue
    os.makedirs(os.path.join(DESTINO, split, "images"))
    os.makedirs(os.path.join(DESTINO, split, "labels"))
    inst, n_img, nulas = Counter(), 0, 0
    for img in sorted(os.listdir(dir_img)):
        base = os.path.splitext(img)[0]
        lbl = os.path.join(ORIGEN, split, "labels", base + ".txt")
        lineas = open(lbl).read().splitlines() if os.path.exists(lbl) else []
        nuevas, excluir = [], False
        for ln in lineas:
            p = ln.split()
            if not p:
                continue
            nuevo = mapa.get(int(p[0]))
            if nuevo is None:
                excluir = True
                break
            es_segmentacion |= len(p) > 5
            nuevas.append((nuevo, " ".join([str(nuevo)] + p[1:])))
        if excluir:
            excluidas.append(f"{split}/{img}")
            continue
        shutil.copy2(os.path.join(dir_img, img), os.path.join(DESTINO, split, "images", img))
        with open(os.path.join(DESTINO, split, "labels", base + ".txt"), "w") as f:
            f.write("\n".join(t for _, t in nuevas) + ("\n" if nuevas else ""))
        inst.update(CLASES[c] for c, _ in nuevas)
        n_img += 1
        nulas += not nuevas
    resumen[split] = {"imagenes": n_img, "fondo_null": nulas, **{c: inst[c] for c in CLASES}}

with open(os.path.join(DESTINO, "data.yaml"), "w") as f:
    yaml.safe_dump({"path": DESTINO, "train": "train/images", "val": "valid/images",
                    "test": "test/images", "names": dict(enumerate(CLASES))}, f, sort_keys=False)

import pandas as pd
tabla = pd.DataFrame(resumen).T
print(tabla.to_string())
print(f"\nImagenes excluidas (Patologias / clases basura): {len(excluidas)}")
for e in excluidas[:20]:
    print("  ", e)
print("\nTipo de etiquetas:", "SEGMENTACION (poligonos)" if es_segmentacion else "CAJAS")
tabla.to_csv("/content/drive/MyDrive/distribucion_dataset_v2.csv")  # evidencia para la tesis

In [ ]:
# 5) Controles de calidad antes de entrenar
tr = resumen["train"]
if tr["fondo_null"] < 150:
    print(f"AVISO: solo {tr['fondo_null']} imagenes de fondo en train. Se recomiendan 150-250 "
          "(cornisas, sombras, juntas marcadas como Null en Roboflow) para evitar el falso positivo de cornisa.")
ratio = tr["humidity"] / max(tr["crack"], 1)
print(f"Relacion humidity/crack en train: {ratio:.1f} : 1")
if ratio > 3:
    print("AVISO: desbalance alto. Conviene etiquetar mas grietas o reducir tiles solo-humedad.")
if resumen.get("test", {}).get("imagenes", 0) < 100:
    print("AVISO: menos de 100 imagenes de test; la tabla de metricas de la tesis sera poco robusta.")

In [ ]:
# 6) ENTRENAMIENTO (mismos hiperparametros que best.pt v1)
from ultralytics import YOLO

TAREA = "segment" if es_segmentacion else "detect"
PESOS = "yolov8s-seg.pt" if TAREA == "segment" else "yolov8s.pt"
NOMBRE = "heritage_damage_v2"

model = YOLO(PESOS)
results = model.train(
    data=f"{DESTINO}/data.yaml",
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    device=0,
    workers=8,
    project=f"runs/{TAREA}",
    name=NOMBRE,
    exist_ok=True,
    optimizer="AdamW",
    seed=42,
    deterministic=True,
    save_period=10,
    # Augmentations orientadas al problema real:
    hsv_v=0.2,      # brillo +-20%: una sombra no es una grieta
    degrees=10.0,   # rotacion +-10 grados
    fliplr=0.5,     # flip horizontal
    flipud=0.0,     # sin flip vertical: la gravedad importa (humedad, desprendimiento)
)
BEST = f"runs/{TAREA}/{NOMBRE}/weights/best.pt"

In [ ]:
# 7) Metricas en TEST (tabla para la tesis) y verificacion de clases
tuned = YOLO(BEST)
assert tuned.names == {0: "crack", 1: "humidity", 2: "spalling"}, tuned.names
print("Clases OK:", tuned.names, "| tarea:", tuned.task)

m = tuned.val(data=f"{DESTINO}/data.yaml", split="test", conf=0.001, iou=0.45)
filas = []
for k, i in enumerate(m.box.ap_class_index):   # solo clases presentes en test
    p, r, ap50, ap = m.box.class_result(k)
    fila = {"clase": tuned.names[int(i)], "precision": p, "recall": r, "mAP50": ap50, "mAP50-95": ap}
    if TAREA == "segment":
        fila["mask_mAP50"] = m.seg.class_result(k)[2]
    filas.append(fila)
metricas = pd.DataFrame(filas)
print(metricas.round(3).to_string(index=False))
print(m.results_dict)
metricas.to_csv("/content/drive/MyDrive/metricas_test_v2.csv", index=False)

In [ ]:
# 8) Copiar best.pt al Drive y descargarlo (reemplaza Heritage_Detector/best.pt tras probarlo)
!cp {BEST} "/content/drive/MyDrive/best_v2.pt"
from google.colab import files
files.download(BEST)